## Comprehensive Hugging Face Tutorial

This notebook provides an exhaustive guide to the **Hugging Face** ecosystem, covering basic to advanced functionalities across its core libraries: `transformers`, `datasets`, `tokenizers`, `evaluate`, `optimum`, and `trl`. It includes tasks like text classification, generative tasks, audio processing, vision tasks, reinforcement learning, and integrations with frameworks like LangChain and LlamaIndex.

## What is Hugging Face?
Hugging Face is an open-source platform offering tools for building, training, and deploying machine learning models. Its key libraries include:
- **transformers**: Pre-trained models for NLP, vision, and audio.
- **datasets**: Efficient dataset loading and processing.
- **tokenizers**: Fast, customizable tokenization.
- **evaluate**: Standardized evaluation metrics.
- **optimum**: Model optimization for inference.
- **trl**: Reinforcement learning for model alignment.

## Objectives
- Cover all major Hugging Face functionalities.
- Provide theoretical context and practical examples.
- Ensure compatibility with Jupyter by fixing JSON formatting issues.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install transformers datasets tokenizers evaluate optimum trl torch torchvision torchaudio sentencepiece accelerate langchain llama-index spacy`
- Sample data directory (`data/`) with `positive.txt`, `negative.txt`, and optionally `audio.wav`.
- Optional: GPU and Hugging Face Hub account.

## Structure
1. Setup and Basic Text Classification
2. Tokenization Basics
3. Dataset Loading and Processing
4. Fine-Tuning a Model
5. Custom Dataset and Training
6. Model Evaluation with Metrics
7. Generative Tasks (Text Generation)
8. Question Answering
9. Named Entity Recognition (NER)
10. Audio Processing (Speech-to-Text, Text-to-Speech)
11. Vision Tasks (Image Classification, Object Detection)
12. Advanced Pipelines (Multimodal)
13. Model Optimization with Optimum
14. Reinforcement Learning with Human Feedback (RLHF)
15. Custom Tokenizer Training
16. Custom Pipeline Creation
17. Integration with LangChain and LlamaIndex
18. Deployment with Hugging Face Hub

Let's dive in!

## 1. Setup and Basic Text Classification

**Theory**: The `transformers` library provides pre-trained models for tasks like text classification. The `pipeline` API simplifies inference.

In [ ]:
# Install required packages
!pip install transformers datasets tokenizers evaluate optimum trl torch torchvision torchaudio sentencepiece accelerate langchain llama-index spacy

In [ ]:
# Basic Text Classification with Pipeline
from transformers import pipeline

# Initialize sentiment analysis pipeline
# Theory: Pipeline abstracts model and tokenizer loading for quick inference
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# Test sentences
texts = [
    "I love using Hugging Face for NLP tasks!",
    "This model is not performing well."
]

# Run inference
results = classifier(texts)
for text, result in zip(texts, results):
    print(f"Text: {text}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

## 2. Tokenization Basics

**Theory**: Tokenization converts text into numerical inputs for models.

In [ ]:
# Basic Tokenization
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Sample text
text = "Hugging Face makes NLP easy and fun!"

# Tokenize
tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

# Decode tokens
decoded_text = tokenizer.decode(tokens['input_ids'][0])

print(f"Original Text: {text}")
print(f"Token IDs: {tokens['input_ids'][0].tolist()}")
print(f"Decoded Text: {decoded_text}")

## 3. Dataset Loading and Processing

**Theory**: The `datasets` library streamlines loading and preprocessing datasets.

In [ ]:
# Dataset Loading and Processing
from datasets import load_dataset

# Load IMDb dataset
dataset = load_dataset("imdb", split="train[:1000]")

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Preprocess function
def preprocess_function(examples):
    """Tokenizes text and prepares labels"""
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

# Apply preprocessing
encoded_dataset = dataset.map(preprocess_function, batched=True)
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Dataset Size: {len(encoded_dataset)}")
print(f"Sample: {encoded_dataset[0]['text'][:100]}...")

## 4. Fine-Tuning a Model

**Theory**: Fine-tuning adapts pre-trained models to specific tasks.

In [ ]:
# Fine-Tuning a Model
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# Load dataset
dataset = load_dataset("imdb", split={"train": "train[:1000]", "test": "test[:200]"})

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Preprocess dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(preprocess_function, batched=True)
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Define training arguments
training_args = TrainingArguments(
    output_dir=".\\results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir=".\\logs",
    logging_steps=10
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"]
)

# Train model
trainer.train()

# Save model
model.save_pretrained(".\\fine_tuned_distilbert")
tokenizer.save_pretrained(".\\fine_tuned_distilbert")

print("Fine-tuning completed!")

## 5. Custom Dataset and Training

**Theory**: Custom datasets can be created from local files.

In [ ]:
# Custom Dataset and Training
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import os

# Load custom dataset
def load_custom_dataset(data_dir="data"):
    texts, labels = [], []
    for label, filename in [(1, "positive.txt"), (0, "negative.txt")]:
        file_path = os.path.join(data_dir, filename)
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    texts.append(line.strip())
                    labels.append(label)
    return Dataset.from_dict({"text": texts, "label": labels})

# Load dataset
dataset = load_custom_dataset()
train_test = dataset.train_test_split(test_size=0.2)

# Initialize tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Preprocess dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = train_test.map(preprocess_function, batched=True)
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Define training arguments
training_args = TrainingArguments(
    output_dir=".\\custom_results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"]
)

# Train model
trainer.train()

print("Custom dataset training completed!")

## 6. Model Evaluation with Metrics

**Theory**: The `evaluate` library offers standardized metrics.

In [ ]:
# Model Evaluation with Metrics
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer
from datasets import load_dataset
import evaluate

# Load fine-tuned model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(".\\fine_tuned_distilbert")
tokenizer = AutoTokenizer.from_pretrained(".\\fine_tuned_distilbert")

# Load test dataset
dataset = load_dataset("imdb", split="test[:200]")

# Preprocess dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(preprocess_function, batched=True)
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Initialize Trainer
trainer = Trainer(model=model)

# Predict
predictions = trainer.predict(encoded_dataset)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Compute metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
accuracy_result = accuracy.compute(predictions=preds, references=labels)
f1_result = f1.compute(predictions=preds, references=labels)

print(f"Accuracy: {accuracy_result['accuracy']:.4f}")
print(f"F1 Score: {f1_result['f1']:.4f}")

## 7. Generative Tasks (Text Generation)

**Theory**: Generative tasks use models like GPT-2 for text completion.

In [ ]:
# Text Generation
from transformers import pipeline

# Initialize text generation pipeline
generator = pipeline("text-generation", model="gpt2")

# Generate text
prompt = "Once upon a time, Hugging Face created"
result = generator(prompt, max_length=50, num_return_sequences=1)

print(f"Prompt: {prompt}")
print(f"Generated: {result[0]['generated_text']}\n")

## 8. Question Answering

**Theory**: Supports extractive and abstractive question answering.

In [ ]:
# Question Answering
from transformers import pipeline

# Extractive QA with BERT
qa_pipeline = pipeline("question-answering", model="bert-large-uncased-whole-word-masking-finetuned-squad")
context = "Hugging Face is a company that provides NLP tools and models."
question = "What does Hugging Face provide?"
result = qa_pipeline(question=question, context=context)
print(f"Extractive QA - Question: {question}")
print(f"Answer: {result['answer']}, Score: {result['score']:.4f}\n")

# Abstractive QA with T5
t5_pipeline = pipeline("text2text-generation", model="t5-small")
input_text = f"question: {question} context: {context}"
result = t5_pipeline(input_text, max_length=50)
print(f"Abstractive QA - Question: {question}")
print(f"Answer: {result[0]['generated_text']}\n")

## 9. Named Entity Recognition (NER)

**Theory**: NER identifies entities in text.

In [ ]:
# Named Entity Recognition
from transformers import pipeline

# Initialize NER pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", grouped_entities=True)

# Test text
text = "Hugging Face is based in New York and was founded by Julien Chaumond."

# Run NER
results = ner_pipeline(text)

print(f"Text: {text}")
for entity in results:
    print(f"Entity: {entity['word']}, Type: {entity['entity_group']}, Score: {entity['score']:.4f}")

## 10. Audio Processing (Speech-to-Text, Text-to-Speech)

**Theory**: Supports audio tasks with Wav2Vec2 and SpeechT5.

In [ ]:
# Audio Processing
from transformers import pipeline, SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
import soundfile as sf
import torch

# Speech-to-Text with Wav2Vec2
stt_pipeline = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h")
audio_path = "data\\audio.wav"
if os.path.exists(audio_path):
    transcription = stt_pipeline(audio_path)
    print(f"Speech-to-Text: {transcription['text']}\n")

# Text-to-Speech with SpeechT5
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

text = "Hugging Face makes AI accessible!"
inputs = processor(text=text, return_tensors="pt")
speaker_embeddings = torch.zeros((1, 512))
speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)

# Save audio
sf.write("output_speech.wav", speech.numpy(), samplerate=16000)
print(f"Text-to-Speech: Audio saved as output_speech.wav")

## 11. Vision Tasks (Image Classification, Object Detection)

**Theory**: Supports vision tasks with ViT and DETR.

In [ ]:
# Vision Tasks
from transformers import pipeline, ViTImageProcessor, ViTForImageClassification, DetrImageProcessor, DetrForObjectDetection
from PIL import Image
import requests
import torch

# Image Classification with ViT
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
image_url = "https://images.unsplash.com/photo-1518791841217-8f162f1e1131"
image = Image.open(requests.get(image_url, stream=True).raw)
results = classifier(image)
print("Image Classification:")
for result in results[:2]:
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")

# Object Detection with DETR
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
target_sizes = torch.tensor([image.size[::-1]])
results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.9)[0]

print("\nObject Detection:")
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    print(f"Label: {model.config.id2label[label.item()]}, Score: {score.item():.4f}, Box: {box.tolist()}")

## 12. Advanced Pipelines (Multimodal)

**Theory**: Multimodal tasks combine text and images.

In [ ]:
# Multimodal Pipeline with CLIP
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests

# Load CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load sample image
url = "https://images.unsplash.com/photo-1518791841217-8f162f1e1131"
image = Image.open(requests.get(url, stream=True).raw)

# Define text labels
labels = ["a cat", "a dog", "a car"]

# Process inputs
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

# Run inference
outputs = model(**inputs)
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)

# Display results
for label, prob in zip(labels, probs[0]):
    print(f"Label: {label}, Probability: {prob:.4f}")

## 13. Model Optimization with Optimum

**Theory**: Optimizes models using quantization and pruning.

In [ ]:
# Model Optimization with Optimum
from optimum.onnxruntime import ORTModelForSequenceClassification
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from transformers import AutoTokenizer

# Load model and tokenizer
model_id = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Convert to ONNX and quantize
model = ORTModelForSequenceClassification.from_pretrained(model_id, export=True)
quantized_model = ORTModelForSequenceClassification.from_pretrained(
    model_id,
    export=True,
    quantization_config=AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=True)
)

# Save quantized model
quantized_model.save_pretrained(".\\quantized_distilbert")
tokenizer.save_pretrained(".\\quantized_distilbert")

# Test quantized model
from optimum.onnxruntime import ORTSeqClassPipeline
pipeline = ORTSeqClassPipeline(model=quantized_model, tokenizer=tokenizer)
text = "Hugging Face is amazing!"
result = pipeline(text)
print(f"Text: {text}\nSentiment: {result[0]['label']}, Score: {result[0]['score']:.4f}")

## 14. Reinforcement Learning with Human Feedback (RLHF)

**Theory**: RLHF aligns models with human preferences.

In [ ]:
# RLHF with TRL
from trl import PPOTrainer, PPOConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Load dataset
dataset = load_dataset("imdb", split="train[:100]")

# Preprocess dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128, return_tensors="pt")

encoded_dataset = dataset.map(preprocess_function, batched=True)

# Define PPO config
config = PPOConfig(
    model_name="gpt2",
    learning_rate=1e-5,
    batch_size=8
)

# Initialize PPO trainer
ppo_trainer = PPOTrainer(config, model, tokenizer=tokenizer)

print("RLHF training setup completed. Actual training requires a reward model.")

## 15. Custom Tokenizer Training

**Theory**: Custom tokenizers optimize for specific domains.

In [ ]:
# Custom Tokenizer Training
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import os

# Prepare text corpus
corpus = []
data_dir = "data"
for filename in os.listdir(data_dir):
    with open(os.path.join(data_dir, filename), "r", encoding="utf-8") as f:
        corpus.append(f.read())

# Initialize tokenizer
tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Define trainer
trainer = trainers.WordPieceTrainer(vocab_size=10000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

# Train tokenizer
tokenizer.train_from_iterator(corpus, trainer)

# Save tokenizer
tokenizer.save(".\\custom_tokenizer.json")

# Test tokenizer
output = tokenizer.encode("Hugging Face is awesome!")
print(f"Tokens: {output.tokens}")
print(f"Token IDs: {output.ids}")

## 16. Custom Pipeline Creation

**Theory**: Custom pipelines allow task-specific processing.

In [ ]:
# Custom Pipeline Creation
from transformers import Pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Define custom pipeline
class CustomSentimentPipeline(Pipeline):
    def _sanitize_parameters(self, **kwargs):
        preprocess_kwargs = {}
        if "max_length" in kwargs:
            preprocess_kwargs["max_length"] = kwargs["max_length"]
        return preprocess_kwargs, {}, {}

    def preprocess(self, inputs, max_length=128):
        return self.tokenizer(inputs, padding=True, truncation=True, max_length=max_length, return_tensors="pt")

    def _forward(self, model_inputs):
        return self.model(**model_inputs)

    def postprocess(self, model_outputs):
        logits = model_outputs.logits
        probs = logits.softmax(dim=-1)
        return [{"label": "POSITIVE" if prob.argmax().item() == 1 else "NEGATIVE", "score": prob.max().item()} for prob in probs]

# Initialize pipeline
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
custom_pipeline = CustomSentimentPipeline(model=model, tokenizer=tokenizer)

# Test pipeline
texts = ["Hugging Face is great!", "This is terrible."]
results = custom_pipeline(texts)
for text, result in zip(texts, results):
    print(f"Text: {text}\nSentiment: {result['label']}, Score: {result['score']:.4f}\n")

## 17. Integration with LangChain and LlamaIndex

**Theory**: Integrates with LangChain and LlamaIndex for RAG.

In [ ]:
# Integration with LangChain and LlamaIndex
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from transformers import pipeline

# LangChain Integration
hf_pipeline = pipeline("text-generation", model="gpt2", max_length=50)
llm = HuggingFacePipeline(pipeline=hf_pipeline)
prompt = PromptTemplate(input_variables=["question"], template="Answer: {question}")
chain = prompt | llm
response = chain.invoke({"question": "What is Hugging Face?"})
print(f"LangChain Response: {response}\n")

# LlamaIndex Integration
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
documents = SimpleDirectoryReader(input_dir="data").load_data()
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
query_engine = index.as_query_engine()
response = query_engine.query("What is Hugging Face?")
print(f"LlamaIndex Response: {response}")

## 18. Deployment with Hugging Face Hub

**Theory**: The Hugging Face Hub enables model sharing.

In [ ]:
# Deployment with Hugging Face Hub
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from huggingface_hub import login

# Log in to Hugging Face Hub
login(token="your_huggingface_token")

# Load fine-tuned model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(".\\fine_tuned_distilbert")
tokenizer = AutoTokenizer.from_pretrained(".\\fine_tuned_distilbert")

# Push to Hub
model.push_to_hub("my-fine-tuned-distilbert")
tokenizer.push_to_hub("my-fine-tuned-distilbert")

# Test inference from Hub
classifier = pipeline("sentiment-analysis", model="your_username/my-fine-tuned-distilbert")
text = "I love Hugging Face!"
result = classifier(text)
print(f"Text: {text}\nSentiment: {result[0]['label']}, Score: {result[0]['score']:.4f}")

## Conclusion

This notebook covers the Hugging Face ecosystem comprehensively, from basic to advanced functionalities. For further exploration, refer to the [Hugging Face Documentation](https://huggingface.co/docs).